In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 10_inference_pipeline_production (VERSIÓN SIMPLIFICADA)
# MAGIC 
# MAGIC **SOLUCIÓN PARA CLUSTERS SIN ACCESO AL SISTEMA DE ARCHIVOS LOCAL**
# MAGIC 
# MAGIC Esta versión carga artefactos desde Delta/CSV en lugar de .pkl
# MAGIC y reconstruye los objetos StandardScaler y PCA

# COMMAND ----------

import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

SILVER_PATH = "/Volumes/olist/olist_silver/silver/"
MODELS_PATH = "/Volumes/olist/olist_gold/models/"
INFERENCE_PATH = "/Volumes/olist/olist_gold/inference/"
GOLD_PATH = "/Volumes/olist/olist_gold/gold/"

START_DATE = "2018-10-01 00:00:00"
END_DATE = "2018-10-17 17:30:18"
CUTOFF_DATE = "2018-10-17 23:59:59"

print("=" * 80)
print("🚀 PIPELINE DE INFERENCIA (VERSIÓN SIMPLIFICADA)")
print("=" * 80)
print(f"\n📅 Periodo: {START_DATE} → {END_DATE}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 0: Verificación y Carga de Artefactos

# COMMAND ----------

print("🔍 ETAPA 0: CARGANDO ARTEFACTOS\n" + "="*80 + "\n")

# Cargar metadata
try:
    metadata = spark.read.format("delta").load(f"{MODELS_PATH}transformation_metadata/").toPandas()
    n_pca_expected = int(metadata['n_pca_components'].iloc[0])
    print(f"✅ Metadata cargada:")
    print(f"   • Componentes PCA: {n_pca_expected}")
    print(f"   • Varianza: {float(metadata['pca_variance_explained'].iloc[0])*100:.2f}%\n")
except Exception as e:
    print(f"❌ Error: {e}\n⚠️  Ejecuta: 06b_pca_save_simple")
    raise

# Cargar features retenidas
features_retained_df = spark.read.format("delta").load(f"{MODELS_PATH}features_retained/").toPandas()
features_retained = features_retained_df.sort_values('order')['feature'].tolist()
print(f"✅ Features retenidas: {len(features_retained)}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 1: Extracción de Datos

# COMMAND ----------

print("📥 ETAPA 1: EXTRACCIÓN DE DATOS\n" + "="*80 + "\n")

orders_full = spark.read.format("delta").load(f"{SILVER_PATH}orders_full/")

orders_production = orders_full.filter(
    (F.col("order_purchase_timestamp") >= F.lit(START_DATE)) &
    (F.col("order_purchase_timestamp") <= F.lit(END_DATE)) &
    (F.col("order_status") != "canceled") &
    (F.col("customer_id").isNotNull())
)

n_orders = orders_production.count()
n_customers = orders_production.select("customer_id").distinct().count()

print(f"✅ Órdenes: {n_orders:,}")
print(f"✅ Clientes: {n_customers:,}\n")

if n_orders == 0:
    raise ValueError("No hay órdenes en el periodo")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 2: Generación de Features

# COMMAND ----------

print("🎯 ETAPA 2: GENERACIÓN DE FEATURES\n" + "="*80 + "\n")

features = orders_production.groupBy("customer_id").agg(
    # RFM
    F.datediff(F.lit(CUTOFF_DATE), F.max("order_purchase_timestamp")).alias("recency"),
    F.count("order_id").alias("frequency"),
    F.sum("payment_sum").alias("monetary"),
    # Tickets
    F.avg("payment_sum").alias("avg_ticket"),
    F.max("payment_sum").alias("max_ticket"),
    F.min("payment_sum").alias("min_ticket"),
    F.stddev("payment_sum").alias("std_ticket"),
    # Items
    F.avg("items_count").alias("avg_items_per_order"),
    F.max("items_count").alias("max_items_per_order"),
    F.sum("items_count").alias("total_items"),
    F.avg("distinct_products").alias("avg_distinct_products"),
    F.sum("distinct_products").alias("total_distinct_products"),
    # Precios y flete
    F.avg("sum_price").alias("avg_price"),
    F.sum("sum_price").alias("total_price"),
    F.avg("sum_freight").alias("avg_freight"),
    F.sum("sum_freight").alias("total_freight"),
    # Pagos
    F.avg("avg_installments").alias("avg_installments"),
    F.max("avg_installments").alias("max_installments"),
    F.avg("n_payment_types").alias("avg_payment_types"),
    # Reviews
    F.avg("avg_review_score").alias("avg_review_score"),
    F.min("avg_review_score").alias("min_review_score"),
    F.max("avg_review_score").alias("max_review_score"),
    F.count(F.when(F.col("avg_review_score").isNotNull(), 1)).alias("orders_with_review"),
    # Temporales
    F.min("order_purchase_timestamp").alias("first_purchase"),
    F.max("order_purchase_timestamp").alias("last_purchase"),
    F.datediff(F.max("order_purchase_timestamp"), F.min("order_purchase_timestamp")).alias("customer_lifetime_days"),
    # Entrega
    F.avg(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("avg_delivery_days"),
    F.max(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("max_delivery_days"),
    F.avg(F.datediff("order_delivered_customer_date", "order_estimated_delivery_date")).alias("avg_delay_days"),
    F.count(F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1)).alias("delayed_orders"),
    # Status
    F.count(F.when(F.col("order_status") == "delivered", 1)).alias("delivered_orders"),
    F.count(F.when(F.col("order_status") == "shipped", 1)).alias("shipped_orders")
)

# Features temporales
features = features \
    .withColumn("first_purchase_month", F.month("first_purchase")) \
    .withColumn("first_purchase_day", F.dayofmonth("first_purchase")) \
    .withColumn("first_purchase_dow", F.dayofweek("first_purchase")) \
    .withColumn("last_purchase_month", F.month("last_purchase")) \
    .withColumn("last_purchase_day", F.dayofmonth("last_purchase")) \
    .withColumn("last_purchase_dow", F.dayofweek("last_purchase")) \
    .drop("first_purchase", "last_purchase")

# Features de interacción
features = features \
    .withColumn("freight_price_ratio", 
                F.when(F.col("total_price") != 0, F.col("total_freight") / F.col("total_price")).otherwise(0)) \
    .withColumn("monetary_per_order", 
                F.when(F.col("frequency") != 0, F.col("monetary") / F.col("frequency")).otherwise(0)) \
    .withColumn("items_per_monetary", 
                F.when(F.col("monetary") != 0, F.col("total_items") / F.col("monetary")).otherwise(0)) \
    .withColumn("products_per_order", 
                F.when(F.col("frequency") != 0, F.col("total_distinct_products") / F.col("frequency")).otherwise(0)) \
    .withColumn("review_score_x_monetary", F.col("avg_review_score") * F.col("monetary")) \
    .withColumn("delayed_ratio", 
                F.when(F.col("frequency") != 0, F.col("delayed_orders") / F.col("frequency")).otherwise(0)) \
    .withColumn("delivered_ratio", 
                F.when(F.col("frequency") != 0, F.col("delivered_orders") / F.col("frequency")).otherwise(0)) \
    .withColumn("orders_per_day", 
                F.when(F.col("customer_lifetime_days") != 0, F.col("frequency") / F.col("customer_lifetime_days")).otherwise(0))

customer_features_raw = features.fillna(0).toPandas()

print(f"✅ Features generadas: {customer_features_raw.shape}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 3: Selección de Features

# COMMAND ----------

print("✂️  ETAPA 3: SELECCIÓN DE FEATURES\n" + "="*80 + "\n")

customer_ids = customer_features_raw['customer_id'].copy()
feature_cols_raw = [c for c in customer_features_raw.columns if c != 'customer_id']

# Validar features
missing = set(features_retained) - set(feature_cols_raw)
if missing:
    raise ValueError(f"Features faltantes: {missing}")

# Alinear columnas
for col in features_retained:
    if col not in customer_features_raw.columns:
        customer_features_raw[col] = 0

customer_features_selected = customer_features_raw[features_retained].copy()

print(f"✅ Features seleccionadas: {customer_features_selected.shape}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 4: Reconstruir y Aplicar StandardScaler

# COMMAND ----------

print("📏 ETAPA 4: ESTANDARIZACIÓN\n" + "="*80 + "\n")

# Cargar parámetros del scaler
scaler_params_df = spark.read.format("delta").load(f"{MODELS_PATH}scaler_params/").toPandas()
scaler_params_df = scaler_params_df.sort_values('feature_index')

# Reconstruir StandardScaler
scaler = StandardScaler()
scaler.mean_ = scaler_params_df['mean'].values
scaler.scale_ = scaler_params_df['scale'].values
scaler.var_ = scaler_params_df['var'].values
scaler.n_features_in_ = len(scaler_params_df)

print(f"✅ Scaler reconstruido:")
print(f"   Features: {scaler.n_features_in_}")

# Validar
if scaler.n_features_in_ != customer_features_selected.shape[1]:
    raise ValueError(f"Dimensiones no coinciden: {scaler.n_features_in_} vs {customer_features_selected.shape[1]}")

# Aplicar transformación
customer_features_scaled = scaler.transform(customer_features_selected)

print(f"✅ Estandarizado: {customer_features_scaled.shape}")
print(f"   Mean: {customer_features_scaled.mean():.6f}")
print(f"   Std: {customer_features_scaled.std():.6f}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 5: Reconstruir y Aplicar PCA

# COMMAND ----------

print("🔬 ETAPA 5: PCA\n" + "="*80 + "\n")

# Cargar componentes principales
pca_components_df = spark.read.format("delta").load(f"{MODELS_PATH}pca_components/").toPandas()
pca_components_df = pca_components_df.sort_values('component_id')

# Cargar parámetros
pca_params_df = spark.read.format("delta").load(f"{MODELS_PATH}pca_params/").toPandas()
pca_params_df = pca_params_df.sort_values('component_id')

# Cargar mean
pca_mean_df = spark.read.format("delta").load(f"{MODELS_PATH}pca_mean/").toPandas()

# Reconstruir PCA
pca = PCA(n_components=len(pca_params_df))

# Extraer matriz de componentes
component_cols = [c for c in pca_components_df.columns if c != 'component_id']
pca.components_ = pca_components_df[component_cols].values

# Asignar parámetros
pca.explained_variance_ = pca_params_df['explained_variance'].values
pca.explained_variance_ratio_ = pca_params_df['explained_variance_ratio'].values
pca.singular_values_ = pca_params_df['singular_values'].values
pca.mean_ = pca_mean_df['pca_mean'].values
pca.n_features_in_ = len(pca.mean_)
pca.n_components_ = len(pca.components_)

print(f"✅ PCA reconstruido:")
print(f"   Componentes: {pca.n_components_}")
print(f"   Varianza: {pca.explained_variance_ratio_.sum()*100:.2f}%")

# Aplicar transformación
X_pca = pca.transform(customer_features_scaled)

print(f"✅ Transformado: {X_pca.shape}\n")

# Crear DataFrame final
pca_cols = [f'pca_{i+1}' for i in range(pca.n_components_)]
customer_features_pca = pd.DataFrame(X_pca, columns=pca_cols)
customer_features_pca['customer_id'] = customer_ids.values

print(f"✅ Dataset final: {customer_features_pca.shape}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 6: Validación y Persistencia

# COMMAND ----------

print("✅ ETAPA 6: VALIDACIÓN\n" + "="*80 + "\n")

# Validaciones
assert customer_features_pca.shape[1] == pca.n_components_ + 1
assert customer_features_pca.isnull().sum().sum() == 0
assert customer_features_pca['customer_id'].nunique() == len(customer_features_pca)

print("✓ Shape correcto")
print("✓ Sin NaN")
print("✓ IDs únicos\n")

# Guardar
output_path = f"{INFERENCE_PATH}customer_features_pca_20181001_20181017/"

try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.inference")
except:
    pass

spark.createDataFrame(customer_features_pca).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(output_path)

print(f"✅ Guardado: {output_path}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Resumen Final

# COMMAND ----------

print("\n" + "="*80)
print("✅ PIPELINE COMPLETADO")
print("="*80)
print(f"\n📊 Órdenes: {n_orders:,}")
print(f"📊 Clientes: {n_customers:,}")
print(f"📊 Features: {len(features_retained)} → {pca.n_components_} PCA")
print(f"📊 Varianza: {pca.explained_variance_ratio_.sum()*100:.2f}%")
print(f"\n💾 Output: {output_path}")
print(f"\n🎯 Listo para predicción con modelo MLflow")
print("\n" + "="*80)